# Deduplication Evaluation

Calls `/dedup/bootstrap` to reset indexes, then `/dedup/check` to submit each article pair and measures:
- **Duplicate recall** — pairs that should be flagged are flagged
- **False positive rate** — pairs that should NOT be flagged are not flagged
- **Stage breakdown** — how many duplicates caught by MinHash vs embedding

**Prerequisite:** NLP service running at `http://localhost:8001` (`/readyz` returns 200).

In [1]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('dedup_cases.json')
print(f'Loaded {len(cases)} test groups')

Loaded 8 test groups


In [3]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, 'Service not ready'

In [13]:
# Reset dedup indexes before each run to ensure deterministic results
reset_resp = requests.post(f'{NLP_BASE_URL}/dedup/bootstrap', json={'articles': []}, headers=HEADERS)
assert reset_resp.status_code == 200, f'Bootstrap failed: {reset_resp.text}'
print('Dedup indexes reset:', reset_resp.json())

Dedup indexes reset: {'processed': 0, 'duplicates_found': 0, 'indexed': 0}


In [14]:
group_results = []

for case in cases:
    # Fresh indexes per group
    requests.post(f'{NLP_BASE_URL}/dedup/bootstrap', json={'articles': []}, headers=HEADERS)

    expected_pairs = {tuple(sorted(p)) for p in case.get('expected_pairs', [])}
    articles = case['articles']
    found_pairs = set()
    call_log = []

    for art in articles:
        t0 = time.monotonic()
        resp = requests.post(
            f'{NLP_BASE_URL}/dedup-check',
            json={'article_id': art['article_id'], 'text': art['text']}, 
            headers=HEADERS
        )
        lat = time.monotonic() - t0
        assert resp.status_code == 200
        data = resp.json()
        call_log.append({'article_id': art['article_id'], **data, 'latency_s': lat})

        if data['duplicate_of'] is not None:
            pair = tuple(sorted([art['article_id'], data['duplicate_of']]))
            found_pairs.add(pair)

    true_pos  = found_pairs & expected_pairs
    false_pos = found_pairs - expected_pairs
    false_neg = expected_pairs - found_pairs

    overall_pass = (false_pos == set() and false_neg == set())
    icon = '✅' if overall_pass else '❌'

    group_results.append({
        'id': case['id'],
        'description': case['description'],
        'expected_pairs': expected_pairs,
        'found_pairs': found_pairs,
        'true_pos': len(true_pos),
        'false_pos': len(false_pos),
        'false_neg': len(false_neg),
        'passed': overall_pass,
        'call_log': call_log,
    })

    print(f"{icon} [{case['id']}] {case['description']}")
    for log in call_log:
        dup = log['duplicate_of']
        stage = log.get('stage', '-')
        score = log.get('score') or 0.0
        idx = log['indexed']
        print(f"   {log['article_id']}: dup={dup!r}  stage={stage}  score={score:.3f}  indexed={idx}  {log['latency_s']:.2f}s")
    if false_pos:
        print(f"   ⚠ False positives: {false_pos}")
    if false_neg:
        print(f"   ⚠ Missed pairs: {false_neg}")
    print()

✅ [dedup-001] Exact duplicate
   dedup-001-a: dup=None  stage=None  score=0.000  indexed=True  0.22s
   dedup-001-b: dup='dedup-001-a'  stage=minhash  score=1.000  indexed=False  0.22s

✅ [dedup-002] Near-duplicate: different headline/byline, same body
   dedup-002-a: dup=None  stage=None  score=0.000  indexed=True  0.24s
   dedup-002-b: dup='dedup-002-a'  stage=embedding  score=0.997  indexed=False  0.21s

✅ [dedup-003] Paraphrase / rewrite — same facts, different wording
   dedup-003-a: dup=None  stage=None  score=0.000  indexed=True  0.19s
   dedup-003-b: dup='dedup-003-a'  stage=embedding  score=0.900  indexed=False  0.18s

✅ [dedup-004] Clearly distinct articles — same topic, different events
   dedup-004-a: dup=None  stage=None  score=0.000  indexed=True  0.18s
   dedup-004-b: dup=None  stage=None  score=0.000  indexed=True  0.66s

✅ [dedup-005] Same article seen again after indexing (idempotency)
   dedup-005-a: dup=None  stage=None  score=0.000  indexed=True  0.25s
   dedup-005

In [15]:
total_expected = sum(len(g['expected_pairs']) for g in group_results)
total_found_tp = sum(g['true_pos'] for g in group_results)
total_fp = sum(g['false_pos'] for g in group_results)
total_fn = sum(g['false_neg'] for g in group_results)

recall = total_found_tp / total_expected if total_expected else 1.0
precision = total_found_tp / (total_found_tp + total_fp) if (total_found_tp + total_fp) else 1.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print_scorecard('DEDUPLICATION', {
    'Groups': len(group_results),
    'Passing groups': f"{sum(1 for g in group_results if g['passed'])}/{len(group_results)}",
    'Expected duplicate pairs': total_expected,
    'True positives': total_found_tp,
    'False positives': total_fp,
    'False negatives (missed)': total_fn,
    'Recall (target ≥ 0.90)': recall,
    'Precision': precision,
    'F1': f1,
})


  DEDUPLICATION
  Groups                              8
  Passing groups                      8/8
  Expected duplicate pairs            7
  True positives                      7
  False positives                     0
  False negatives (missed)            0
  Recall (target ≥ 0.90)              1.000
  Precision                           1.000
  F1                                  1.000



## Tuning Guide

There are two independent thresholds, set via environment variables:

| Symptom | Lever | Env var | Direction |
|---------|-------|---------|----------|
| Near-duplicate rewrites not caught | Embedding threshold too high | `DEDUP_EMBED_THRESHOLD` | **Lower** (e.g. 0.85 → 0.80) |
| Unrelated articles flagged as dups | Embedding threshold too low | `DEDUP_EMBED_THRESHOLD` | **Raise** (e.g. 0.85 → 0.90) |
| Exact/near-exact not caught | MinHash threshold too high | `DEDUP_LSH_THRESHOLD` | **Lower** (e.g. 0.90 → 0.85) |
| Minor edits flagged as dups | MinHash threshold too low | `DEDUP_LSH_THRESHOLD` | **Raise** (e.g. 0.90 → 0.95) |
| Very short articles not matched | Few shingles — MinHash unreliable | Use embedding-only for len < 50 words | Modify `service.py` |

**How to test threshold changes:**
1. Update `DEDUP_LSH_THRESHOLD` / `DEDUP_EMBED_THRESHOLD` in `docker-compose.yml`
2. Restart: `docker compose restart nlp-service`
3. Re-run this notebook (bootstrap resets state automatically)